In [ ]:
import pandas as pd

listings = pd.read_csv('data/listings_clean.csv', parse_dates=['first_review', 'last_review'])
print(listings.shape)

Claim 1: "A night in Stockholm costs 1,791."

In [12]:
# Check the median price and how many listings have a price
print(listings['price'].median())
# 1,791 is the median, that is true

print(listings['price'].isna().sum(), (listings['price'].isna().mean() * 100).round(1))
# 35% of listings do not have the price 

# Check if there is the connection between price and availability
print(pd.crosstab(listings['price'].isna(), listings['availability_365'] == 0))
# 1490 listings (from 1617) are not available at all

# Compare long stays (28+ nights) with normal short stays
has_price = listings[listings['price'].notna()]
long_stay = has_price['minimum_nights'] >= 28

print(f"\nListings with price: {listings['price'].notna().sum()}\n")
print(has_price.groupby(long_stay)['price'].agg(['count', 'median']).round(1))


1791.0
1617 35.4
availability_365  False  True 
price                         
False              2953      4
True                127   1490

Listings with price: 2957

                count  median
minimum_nights               
False            2860  1839.2
True               97   708.9


Verdict: Holds with reservations.

The median price is 1791, so the number is right, and the data dictionary says the price is in SEK. However, 1617 listings (35.4%) have no price at all, and almost all of them have 0 available days in the year. Also 97 listings with a price require at least 28 nights and have a much lower median price of 708.9 SEK, while the other 2860 normal listings have a median of 1839.2 SEK.

New sentence: "The median price per night is 1791 SEK for the 2957 listings in Stockholm that have a price (65% of all listings), or 1839.2 SEK if we do not include long stays of 28 nights or more."

Claim 2: "The listings with a perfect 5.00 rating are the best in Stockholm."

In [ ]:
# Use only listings with reviews
rated = listings[listings['has_reviews']]

# Mark 'perfect' listings
perfect = rated['review_scores_rating'] == 5.00

print(len(rated))
print(perfect.sum(), (perfect.mean() * 100).round(1))

# See how many are 'perfect' and have at least 20 reviews at the same time
print((perfect & (rated['number_of_reviews'] >= 20)).sum())

print(rated.groupby(perfect)['number_of_reviews'].agg(['count', 'median']))

3930
1484 37.8
130
                      count  median
review_scores_rating               
False                  2446    25.0
True                   1484     4.0


Verdict: Does not hold.

Out of 3930 listings with reviews, 1484 have perfect 5.00 rating (37.8%). However only 130 of these perfect listings have 20 or more reviews (8.8%), which means 91.2% of them have fewer than 20 reviews. Also the median number of reviews for 5.00 listings is only 4, while for other rated listings it is 25. Having a 5.00 score mostly shows that a listing has very few reviews, not that it is the best.

New sentence: "38% of the rated listings in Stockholm have a perfect 5.00 rating, but 91% of them have fewer than 20 reviews (with a median of 4 reviews), so the rating does not show which listings are the best."

Claim 3: "The market has grown: only 131 listings started in 2016, but 610 in 2024."

In [ ]:
# Get the year of the first review for each listing
first_year = listings['first_review'].dt.year

print((first_year == 2016).sum())
print((first_year == 2024).sum())
# Numbers are the same as in the claim

# Check how many listings from each year still got reviews in 2026
reviewed_2026 = listings['last_review'].dt.year == 2026
print(reviewed_2026.groupby(first_year).agg(['count', 'mean']).round(2))

131
610
              count  mean
first_review             
2011.0            2  1.00
2012.0           12  0.58
2013.0           26  0.58
2014.0           51  0.65
2015.0          116  0.53
2016.0          131  0.53
2017.0          148  0.53
2018.0          191  0.51
2019.0          234  0.55
2020.0           96  0.54
2021.0          159  0.58
2022.0          332  0.52
2023.0          565  0.59
2024.0          610  0.67
2025.0          542  0.80
2026.0          715  1.00


Verdict: Cannot be concluded from this table.

The numbers are right: 131 listings in the table got their first review in 2016, and 610 got their first review in 2024. However this table only includes listings that still existed when the data was collected in September 2026. Listings that started in 2016 and closed later are missing from the table (survivorship bias), so we cannot see how many listings actually started in 2016 or whether the market grew.

New sentence: "Of the Stockholm Airbnb listings that still existed in September 2026, 131 received their first review in 2016 and 610 in 2024, so we cannot say from this table how many listings started in those years."

Claim 4: "Most hosts in Stockholm are professional companies with many listings."

In [ ]:
# Compare total listings and unique hosts
print("Total listings:", len(listings))
print("Unique hosts:", listings['host_id'].nunique())

# Count how many listings each host has
host_counts = listings['host_id'].value_counts()
# print(host_counts)

# Hosts with 1 listing vs hosts with 2 or more listings
print("1 listing:", (host_counts == 1).sum(), ((host_counts == 1).mean() * 100).round(1))
print("2+ listings:", (host_counts >= 2).sum(), ((host_counts >= 2).mean() * 100).round(1))

# How many listings belong to hosts with 2 or more listings
print("Listings of 2+ hosts:", host_counts[host_counts >= 2].sum(), (host_counts[host_counts >= 2].sum() / len(listings) * 100).round(1))
print("Max listings per host:", host_counts.max())

Total listings: 4574
Unique hosts: 3591
host_id
271346174              49
410412637              37
60382478               24
259019701              21
135323554              20
                       ..
33414364                1
713035605               1
430540839               1
672067666               1
1729223981108209154     1
Name: count, Length: 3591, dtype: int64


Verdict: Does not hold (and whether hosts are companies cannot be concluded from this table).

There are 4574 listings in the table owned by 3591 unique hosts. 3237 (90.1%) hosts have only 1 listing, while only 354 hosts (9.9%) have two or more listings. Those 9.9% own 1337 listings together (29.2% of all listings), but they are still a small part of hosts. Also the table has no column that tells us whether a host is a professional company or a private person.

New sentence: "In Stockholm, 90% of the 3591 Airbnb hosts have only one listing, while the 10% of hosts with two or more listings own 29% of all listings."

Task 5 is different: a data note, not a claim

This table is a snapshot of 4574 Airbnb listings in Stockholm collected by Inside Airbnb on 26 and 27 September 2026 under a CC BY 4.0 license (source: insideairbnb.com). A lot of information is missing: 35.4% of listings have no price (mostly because they have zero available days), and 14.1% have no rating because they have no reviews. Because the table shows only active Airbnb listings at that moment, it does not include older listings that closed before September 2026. Columns starting with estimated_ (such as estimated_occupancy_l365d and estimated_revenue_l365d) are based on assumptions not on real measurements. Because of this the table cannot tell us why the price is high or low, whether guests were truly satisfied, or if a host is a company or a private person. If I read Claim 3 ("The market has grown: only 131 listings started in 2016, but 610 in 2024") in a newspaper, I would most want to correct it because the table is missing all old listings that closed before 2026, so those numbers do not show real market growth.